# Surrogate-1 v2 — Colab T4 fine-tune

**Free GPU**: Google Colab T4 16GB · ~12-15h session · auto-saves to GDrive every 30 min.

**Why Colab?** Kaggle phone-locked (can't sign up new account); user reserves OCI/GCP/Civo for runtime, not training.

Pulls latest `axentx/surrogate-1-pairs-C` (~2,647 shards, 1M+ pairs across coding/dialog/commits/reasoning/iac), runs LoRA SFT on Qwen2.5-Coder-7B base (4-bit qlora, ~6 GB GPU RAM) for 1 epoch, pushes adapter to `axentx/surrogate-1-coder-7b-lora-v2` on HF Hub.

Run order: cell 1 (env) → cell 2 (deps) → cell 3 (data) → cell 4 (model) → cell 5 (train) → cell 6 (push).

In [ ]:
# Cell 1 — env + secrets
import os
from google.colab import userdata

# Set in Colab: 🔑 secrets panel (left sidebar) → add HF_TOKEN, WANDB_API_KEY (optional)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY', '')

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print(f'cuda: {torch.cuda.is_available()}  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# Cell 2 — deps (~3 min)
!pip install -q -U bitsandbytes==0.45.0 transformers==4.47.0 accelerate==1.2.0 peft==0.14.0 trl==0.13.0 datasets==3.2.0 huggingface_hub==0.27.0
!pip install -q wandb

In [ ]:
# Cell 3 — load training corpus from HF Hub (streaming so we don't OOM disk)
from datasets import load_dataset

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
PAIR_DATASETS = [
    'axentx/surrogate-1-pairs-C',  # ~1M+ pairs (coding/dialog/commits/reasoning/iac)
    'axentx/surrogate-1-pairs-B',  # earlier shards
    'axentx/surrogate-1-pairs-A',
]

from datasets import concatenate_datasets, interleave_datasets
shards = []
for ds_id in PAIR_DATASETS:
    try:
        ds = load_dataset(ds_id, split='train', streaming=True)
        shards.append(ds)
        print(f'  ✓ {ds_id}')
    except Exception as e:
        print(f'  ✗ {ds_id}: {e}')
raw = interleave_datasets(shards) if shards else None
assert raw is not None, 'no datasets loaded — check HF_TOKEN'
print('first sample:', next(iter(raw)))

In [ ]:
# Cell 4 — load 4-bit base + apply LoRA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb,
    device_map='auto', trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Cell 5 — SFT trainer (1 epoch, ~6-10h on T4 for ~50k samples)
from trl import SFTConfig, SFTTrainer

def fmt(row):
    p = row.get('prompt') or row.get('instruction') or ''
    r = row.get('response') or row.get('chosen') or row.get('output') or ''
    return {'text': f'<|im_start|>user\n{p}<|im_end|>\n<|im_start|>assistant\n{r}<|im_end|>'}

ds = raw.map(fmt).filter(lambda r: len(r['text']) > 50).take(50_000)

cfg = SFTConfig(
    output_dir='/content/v2-out',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=2,
    optim='paged_adamw_8bit',
    warmup_ratio=0.03,
    max_seq_length=2048,
    dataset_text_field='text',
    report_to=['wandb'] if os.environ.get('WANDB_API_KEY') else 'none',
    run_name=f'surrogate-v2-colab',
)
trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg, tokenizer=tok)
trainer.train()

In [ ]:
# Cell 6 — push adapter to HF Hub
ADAPTER_REPO = 'axentx/surrogate-1-coder-7b-lora-v2'
trainer.model.push_to_hub(ADAPTER_REPO, private=False)
tok.push_to_hub(ADAPTER_REPO)
print(f'✓ adapter pushed to https://huggingface.co/{ADAPTER_REPO}')
print('next: update axentx-pipeline._STRONG_CHAIN to add this LoRA via HF Inference Endpoint')